In [2]:
import sys
sys.path.append(r'C:\Users\Sarthak\Documents\ML\fighter-beta\mma\notebooks')

import pickle
import pandas as pd
import numpy as np
from datetime import datetime

DATA_PATH = r'C:\Users\Sarthak\Documents\ML\fighter-beta\mma\notebooks\02_features\data'

# Load everything
fight_features = pickle.load(open(f'{DATA_PATH}/fight_features.pkl', 'rb'))

print(f"✔ Loaded. Shape: {fight_features.shape}")
print(f"✔ Date range: {fight_features['event_date'].min()} to {fight_features['event_date'].max()}")

✔ Loaded. Shape: (2837, 386)
✔ Date range: 2014-01-15 to 2026-02-07


In [3]:
# ── Build target variable ─────────────────────────────────────────────────
def map_method(method):
    if pd.isna(method): return None
    m = str(method).upper()
    if 'KO' in m or 'TKO' in m: return 'KO/TKO'
    if 'SUB' in m: return 'Submission'
    if 'DEC' in m: return 'Decision'
    return None

fight_features['method_label'] = fight_features['method'].apply(map_method)

# Drop rows with no method label
df = fight_features[fight_features['method_label'].notna()].copy()

print(f"✔ Total fights after filtering: {len(df)}")
print(f"\nMethod distribution:")
print(df['method_label'].value_counts())
print(f"\nMethod %:")
print(df['method_label'].value_counts(normalize=True).round(3) * 100)

✔ Total fights after filtering: 2837

Method distribution:
method_label
Decision      1243
KO/TKO        1036
Submission     558
Name: count, dtype: int64

Method %:
method_label
Decision      43.8
KO/TKO        36.5
Submission    19.7
Name: proportion, dtype: float64


In [6]:
def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 50, 500),
        'max_depth':        trial.suggest_int('max_depth', 2, 5),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 50),
        'subsample':        trial.suggest_float('subsample', 0.4, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 0.8),
        'gamma':            trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.5, 5.0),
        'objective':        'multi:softprob',
        'num_class':        3,
        'random_state':     42,
    }
    m = XGBClassifier(**params)
    m.fit(X_train, y_train)
    return m.score(X_val, y_val)

print("✓ Starting Optuna search (200 trials)...")
import time
start = time.time()
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=200)
print(f"Done in {time.time()-start:.1f}s")

best = study.best_trial
print(f"\nBest val accuracy: {best.value:.4f} ({best.value*100:.1f}%)")
print(f"Best params:")
for k, v in best.params.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# Retrain with best params
best_method_model = XGBClassifier(
    **best.params,
    objective='multi:softprob',
    num_class=3,
    random_state=42
)
best_method_model.fit(X_train, y_train)

train_acc = best_method_model.score(X_train, y_train)
val_acc   = best_method_model.score(X_val, y_val)
test_acc  = best_method_model.score(X_test, y_test)

print(f"\nTrain: {train_acc:.1%}")
print(f"Val:   {val_acc:.1%}")
print(f"Test:  {test_acc:.1%}")

✓ Starting Optuna search (200 trials)...
Done in 183.0s

Best val accuracy: 0.4973 (49.7%)
Best params:
  n_estimators: 167
  max_depth: 3
  min_child_weight: 13
  subsample: 0.7350
  colsample_bytree: 0.4461
  gamma: 2.1956
  reg_alpha: 2.2418
  reg_lambda: 2.3047

Train: 79.2%
Val:   49.7%
Test:  44.1%


In [7]:
import pandas as pd

feat_imp = pd.Series(
    best_method_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print("Top 20 features:")
print(feat_imp.head(20).to_string())

Top 20 features:
diff_kd_per_min_opp_mad           0.012509
diff_win_ratio                    0.011294
diff_body_lpm_dec_avg             0.011076
diff_r1_kd_per_min_dec_avg        0.010865
diff_sub_avg_dec_avg              0.010779
diff_r1_head_lpm_dec_avg          0.010740
diff_head_lpm_dec_avg             0.010679
diff_kd_per_min_opp_dec_avg       0.010487
diff_td_avg_opp_mad               0.010310
diff_r1_ctrl_per_min_dec_avg      0.010029
diff_distance_acc_adjperf         0.009824
age_ratio                         0.009637
diff_body_acc_dec_avg             0.009586
diff_ground_allowed_adjperf       0.009564
diff_slpm_dec_adjperf             0.009547
diff_ctrl_time_per_min_adjperf    0.009536
diff_r1_clinch_lpm_opp_dec_avg    0.009480
diff_td_defense                   0.009390
diff_leg_land_r1_opp_dec_avg      0.009327
diff_win_adjperf                  0.009234


In [9]:
import sqlite3
DB_PATH = r'C:\Users\Sarthak\Documents\ML\fighter-beta\mma_fighters.db'
conn = sqlite3.connect(DB_PATH)
# ── Method-specific features ──────────────────────────────────────────────
results_raw = pd.read_sql("""
    SELECT
        ff.fighter_id,
        ff.fight_id,
        ff.result,
        f.method
    FROM fight_fighters_v2 ff
    JOIN fights_v2 f ON ff.fight_id = f.fight_id
    WHERE ff.result = 'win'
""", conn)

def get_finish_rates(fighter_id, as_of_date, results_raw):
    prev = results_raw[
        (results_raw['fighter_id'] == fighter_id)
    ]
    if len(prev) == 0:
        return {'ko_rate': 0.0, 'sub_rate': 0.0, 'dec_rate': 0.0, 'finish_rate': 0.0}

    total = len(prev)
    ko  = prev['method'].str.contains('KO|TKO', na=False).sum()
    sub = prev['method'].str.contains('SUB', na=False, case=False).sum()
    dec = prev['method'].str.contains('DEC', na=False, case=False).sum()

    return {
        'ko_rate':     ko / total,
        'sub_rate':    sub / total,
        'dec_rate':    dec / total,
        'finish_rate': (ko + sub) / total
    }

# Add method features to fight_features
print("Adding method-specific features...")
rows = []
for _, fight in df.iterrows():
    f1_id = fight['fighter_1_id']
    f2_id = fight['fighter_2_id']

    f1_rates = get_finish_rates(f1_id, fight['event_date'], results_raw)
    f2_rates = get_finish_rates(f2_id, fight['event_date'], results_raw)

    row = {
        'fight_id':          fight['fight_id'],
        'f1_ko_rate':        f1_rates['ko_rate'],
        'f1_sub_rate':       f1_rates['sub_rate'],
        'f1_dec_rate':       f1_rates['dec_rate'],
        'f1_finish_rate':    f1_rates['finish_rate'],
        'f2_ko_rate':        f2_rates['ko_rate'],
        'f2_sub_rate':       f2_rates['sub_rate'],
        'f2_dec_rate':       f2_rates['dec_rate'],
        'f2_finish_rate':    f2_rates['finish_rate'],
        'diff_ko_rate':      f1_rates['ko_rate']     - f2_rates['ko_rate'],
        'diff_sub_rate':     f1_rates['sub_rate']    - f2_rates['sub_rate'],
        'diff_dec_rate':     f1_rates['dec_rate']    - f2_rates['dec_rate'],
        'diff_finish_rate':  f1_rates['finish_rate'] - f2_rates['finish_rate'],
        'max_ko_rate':       max(f1_rates['ko_rate'], f2_rates['ko_rate']),
        'max_sub_rate':      max(f1_rates['sub_rate'], f2_rates['sub_rate']),
        'max_finish_rate':   max(f1_rates['finish_rate'], f2_rates['finish_rate']),
    }
    rows.append(row)

method_feats = pd.DataFrame(rows)
df = df.merge(method_feats, on='fight_id', how='left')
print(f"✔ Added {len(method_feats.columns)-1} method features")
print(f"✔ New shape: {df.shape}")

Adding method-specific features...
✔ Added 15 method features
✔ New shape: (2837, 403)


In [14]:
# Updated feature cols including submission-specific features
METHOD_FEATURE_COLS = [c for c in df.columns
                if c.startswith('diff_')
                or c in ['age_diff', 'age_ratio', 'reach_diff',
                         'height_diff', 'reach_ratio', 'height_ratio',
                         'f1_ko_rate', 'f1_sub_rate', 'f1_dec_rate', 'f1_finish_rate',
                         'f2_ko_rate', 'f2_sub_rate', 'f2_dec_rate', 'f2_finish_rate',
                         'max_ko_rate', 'max_sub_rate', 'max_finish_rate',
                         'f1_sub_att_per_fight', 'f2_sub_att_per_fight',
                         'f1_td_per_fight', 'f2_td_per_fight',
                         'f1_ctrl_per_fight', 'f2_ctrl_per_fight',
                         'max_sub_att_per_fight', 'max_td_per_fight']]

METHOD_FEATURE_COLS = [c for c in METHOD_FEATURE_COLS
                if 'strike_share'     not in c
                and 'exchange_ratio'  not in c
                and 'targeting_share' not in c]

# Rebuild splits
train = df[df['event_date'] <  '2023-01-01']
val   = df[(df['event_date'] >= '2023-01-01') & (df['event_date'] < '2025-01-01')]
test  = df[df['event_date'] >= '2025-01-01']

X_train_m = train[METHOD_FEATURE_COLS].fillna(0)
X_val_m   = val[METHOD_FEATURE_COLS].fillna(0)
X_test_m  = test[METHOD_FEATURE_COLS].fillna(0)
y_train_m = train['method_encoded']
y_val_m   = val['method_encoded']
y_test_m  = test['method_encoded']

print(f"✔ Feature count: {len(METHOD_FEATURE_COLS)}")

# Retrain
def objective_method(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 50, 500),
        'max_depth':        trial.suggest_int('max_depth', 2, 5),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 50),
        'subsample':        trial.suggest_float('subsample', 0.4, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 0.8),
        'gamma':            trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.5, 5.0),
        'objective':        'multi:softprob',
        'num_class':        3,
        'random_state':     42,
        'scale_pos_weight': 1,
    }
    m = XGBClassifier(**params)
    m.fit(X_train_m, y_train_m)
    return m.score(X_val_m, y_val_m)

print("✓ Starting Optuna search (200 trials)...")
start = time.time()
study_method2 = optuna.create_study(direction='maximize')
study_method2.optimize(objective_method, n_trials=200)
print(f"Done in {time.time()-start:.1f}s")

best_m2 = study_method2.best_trial
best_method_model = XGBClassifier(
    **best_m2.params,
    objective='multi:softprob',
    num_class=3,
    random_state=42
)
best_method_model.fit(X_train_m, y_train_m)

train_acc = best_method_model.score(X_train_m, y_train_m)
val_acc   = best_method_model.score(X_val_m,   y_val_m)
test_acc  = best_method_model.score(X_test_m,  y_test_m)

print(f"\nBest val accuracy: {best_m2.value:.1%}")
print(f"Train: {train_acc:.1%}")
print(f"Val:   {val_acc:.1%}")
print(f"Test:  {test_acc:.1%}")
print(f"Train/Val gap: {(train_acc - val_acc)*100:+.1f}%")

# Classification report
from sklearn.metrics import classification_report, confusion_matrix
y_pred = best_method_model.predict(X_val_m)
print("\nClassification Report:")
print(classification_report(y_val_m, y_pred, target_names=le.classes_))

✔ Feature count: 148
✓ Starting Optuna search (200 trials)...


C:\Users\Sarthak\miniconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [21:20:45] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\Sarthak\miniconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [21:20:48] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\Sarthak\miniconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [21:20:49] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\Sarthak\miniconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [21:20:50] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "scale_pos_weight" } are not used.

  bst.up

Done in 178.8s

Best val accuracy: 59.0%
Train: 60.1%
Val:   59.0%
Test:  55.9%
Train/Val gap: +1.1%

Classification Report:
              precision    recall  f1-score   support

    Decision       0.63      0.71      0.67       247
      KO/TKO       0.55      0.68      0.61       183
  Submission       0.51      0.20      0.29       121

    accuracy                           0.59       551
   macro avg       0.57      0.53      0.52       551
weighted avg       0.58      0.59      0.57       551



In [15]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = best_method_model.predict(X_val_m)

print("Classification Report:")
print(classification_report(y_val_m, y_pred, target_names=le.classes_))

print("\nConfusion Matrix:")
cm = pd.DataFrame(
    confusion_matrix(y_val_m, y_pred),
    index=[f'Actual {c}' for c in le.classes_],
    columns=[f'Pred {c}' for c in le.classes_]
)
print(cm)

Classification Report:
              precision    recall  f1-score   support

    Decision       0.63      0.71      0.67       247
      KO/TKO       0.55      0.68      0.61       183
  Submission       0.51      0.20      0.29       121

    accuracy                           0.59       551
   macro avg       0.57      0.53      0.52       551
weighted avg       0.58      0.59      0.57       551


Confusion Matrix:
                   Pred Decision  Pred KO/TKO  Pred Submission
Actual Decision              176           58               13
Actual KO/TKO                 48          125               10
Actual Submission             54           43               24


In [16]:
from sklearn.utils.class_weight import compute_sample_weight

# Compute sample weights to upweight submissions
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_m)

# Retrain best model with sample weights
best_method_model.fit(X_train_m, y_train_m, sample_weight=sample_weights)

train_acc = best_method_model.score(X_train_m, y_train_m)
val_acc   = best_method_model.score(X_val_m,   y_val_m)

y_pred = best_method_model.predict(X_val_m)
print(f"Val: {val_acc:.1%}")
print("\nClassification Report:")
print(classification_report(y_val_m, y_pred, target_names=le.classes_))

Val: 54.8%

Classification Report:
              precision    recall  f1-score   support

    Decision       0.70      0.58      0.63       247
      KO/TKO       0.55      0.55      0.55       183
  Submission       0.36      0.49      0.42       121

    accuracy                           0.55       551
   macro avg       0.54      0.54      0.53       551
weighted avg       0.57      0.55      0.56       551



In [17]:
best_method_model = XGBClassifier(
    **best_m2.params,
    objective='multi:softprob',
    num_class=3,
    random_state=42
)
best_method_model.fit(X_train_m, y_train_m)
print(f"Val: {best_method_model.score(X_val_m, y_val_m):.1%}")

Val: 59.0%


In [18]:
pickle.dump(best_method_model, open(f'{DATA_PATH}/method_model.pkl', 'wb'))
pickle.dump(METHOD_FEATURE_COLS, open(f'{DATA_PATH}/method_feature_cols.pkl', 'wb'))
pickle.dump(le, open(f'{DATA_PATH}/method_label_encoder.pkl', 'wb'))
print("✔ Method model saved.")
print(f"  Classes: {list(le.classes_)}")
print(f"  Features: {len(METHOD_FEATURE_COLS)}")

✔ Method model saved.
  Classes: ['Decision', 'KO/TKO', 'Submission']
  Features: 148


In [19]:
import os
for f in ['method_model.pkl', 'method_feature_cols.pkl', 'method_label_encoder.pkl']:
    path = f'{DATA_PATH}/{f}'
    print(f"{f}: {'✔' if os.path.exists(path) else '✗'}")

method_model.pkl: ✔
method_feature_cols.pkl: ✔
method_label_encoder.pkl: ✔
